
<center><ins><h1>Recovery Rate Analysis</h1></ins></center>

<h5>Definition:</h5>
<ul>
    <li>Sedimentation is a quiescent process that allows the formed flocs to settle under the influence of gravity.</li> 
    <li>It is calculated by OD750(t0) - OD750(t) / OD750(t0) x 100 and expressed in %.</li> 
    <li>Files are imported over .csv in individual sample files. Sample information from file name and values from file.</li>
</ul>

<center>
<img src="../images/recovery-rate_device.png" alt="Device" width="352" height="300">
<img src="../images/recovery-rate_diagram.png" alt="Diagram" width="387" height="300">
<img src="../images/recovery-rate_example-graph.png" alt="Example-Graph" width="291" height="300">
</center>

###### **Author:** Cedric Hering-Peter
###### **Address:** AG Schulz / Botantical Institut / Am Botanischen Garten 5 / 24118 Kiel

# Packages

In [1]:
# Imports
import os, sys
sys.path.append(os.path.abspath("/Users/cedric/Documents/Career/PhD-Student/Experiments/Data-Science"))
from scripts import data_helper, plot_helper
import pandas as pd

# Variables

In [2]:
META_DATA = pd.read_csv("../data/meta_data.csv")
FIL_TIME_12H = True

# Show numeric output in decimal format e.g., 2.15
pd.options.display.float_format = '{:,.2f}'.format

# Functions

In [3]:
def create_entry(file):
    # Insert sample information in a new empty data frame
    entry = pd.DataFrame(columns = META_DATA["SAMPLE_INFORMATION"].dropna().tolist())
    entry.loc[0] = os.path.basename(file).split(".Probe")[0].split("_")

    # Custom data calculation from current sample
    sample = pd.read_csv(file)[" A"]
    od_mean = sample[:10].mean() * 4 # CAUTION: dilution of 4 not true for all samples
    od_max = sample[:60].max() # NOTE We could consider playing with this value
    od_min = sample[250:310].min() # NOTE We could consider playing with this value
    rrate = round(((od_max - od_min) / od_max) * 100, 2)

    # Add data value columns to entry
    entry["MEAN_OD"] = od_mean
    entry["START_OD"] = od_max
    entry["END_OD"] = od_min
    entry["RRATE_%"] = rrate
    return entry

def custom_rrate_import(files):
    df = pd.DataFrame()
    for file in files:
        # Append each entry to new data frame
        df = pd.concat([df, create_entry(file)], ignore_index = True)
    return df

# Raw Import

In [4]:
# Define data path and get all associated files
files_dir = "/Users/cedric/Documents/Career/PhD-Student/Experiments/Instrument-Data/Photometer_Lambda365plus"
files = data_helper.create_filelist(files_dir, skip = "-OD")

# Import raw data frame
raw_df = custom_rrate_import(files)
# Convert data types
raw_df = data_helper.convert_types(raw_df)
# Append start controls to physiology samples    
raw_df = data_helper.copy_start_control(raw_df, META_DATA)
# Sort data
raw_df = raw_df.sort_values(by = ["EXPERIMENT_NAME", "TIME", "SAMPLE_NAME", "BIO_REP", "TEC_REP"], ignore_index = True)
raw_df.head(3)

,ID,EXPERIMENT_NAME,SAMPLE_NAME,TIME,BIO_REP,TEC_REP,MEAN_OD,START_OD,END_OD,RRATE_%
0,CH230710,AT,0.1,0,1,1,1.43,0.49,0.06,87.71
1,CH230802,AT,0.1,0,2,1,0.62,0.36,0.03,91.62
2,CH230907,AT,0.1,0,3,1,1.57,0.47,0.08,83.62


# Custom Cleaning

In [5]:
clean_df = pd.DataFrame(raw_df)

# Drop duplicate measurements, blanks and tec_rep > 1
clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "5-Second-Run"].index, inplace = True)
clean_df.drop(clean_df[clean_df["SAMPLE_NAME"].str.contains("Blank")].index, inplace = True)
clean_df.drop(clean_df[clean_df["TEC_REP"] != 1].index, inplace = True)

# Lower species count from 7 to 5
clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "C.vulgaris"].index, inplace = True)
clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "P.duplex"].index, inplace = True)

clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "+EPS"].index, inplace = True)

print(f"{raw_df.shape[0] - clean_df.shape[0]} rows and {raw_df.shape[1] - clean_df.shape[1]} cols were cleaned.")

48 rows and 0 cols were cleaned.


# File Export

In [6]:
# Export to raw data to csv-file
export_df = pd.DataFrame(clean_df)
file_name = "../../OUTPUT/Raw-Data/Recovery-Rates_Raw-Data.csv"
export_df.to_csv(file_name, encoding = "utf-8", index = False)

# Plot Visualization 

In [7]:
# Filter time before plotting
plot_df = pd.DataFrame(data_helper.filter(clean_df, time = [12]))

# Iterate through all experiment, visualize and save plots
for exp in META_DATA["EXP_ABBR"].unique():
    sub_df = plot_df[plot_df["EXPERIMENT_NAME"] == exp]
    sub_meta = META_DATA[META_DATA["EXP_ABBR"] == exp][:1].reset_index()
    
    plot_helper.visualize_boxplot(sub_df, sub_meta, # current data subset
                                  show_x_label = True,
                                  y_data = "RRATE_%", 
                                  y_label = "Recovery Rate [%/5\']", # Without unit serves as file name
                                  y_ticks = [0, 110, 10],
                                  y_labelpad = 27)

Boxplot "Species Composition" created and saved.
Boxplot "Salinity Treatment" created and saved.
Boxplot "pH Treatment" created and saved.
Boxplot "Antibiotics Treatment" created and saved.
Boxplot "Light Treatment" created and saved.
Boxplot "Temperature Treatment" created and saved.
Boxplot "Culture Composition" created and saved.
Boxplot "Reflocculation" created and saved.
Boxplot "Bioflocculation" created and saved.


# ARCHIVE: Relative Data Calculation

In [8]:
# # Filter and group by all control-like data and calculate the mean
# rel_plot_df = plot_df
# con_fil = rel_plot_df[(rel_plot_df["SAMPLE_NAME"] == "Control") | (rel_plot_df["SAMPLE_NAME"] == "Algaemix") | (rel_plot_df["SAMPLE_NAME"] == "OECD") | (rel_plot_df["SAMPLE_NAME"] == "CoA")]
# zero_vals = con_fil.groupby(by = ["EXPERIMENT_NAME","SAMPLE_NAME", "TIME"], as_index = False).agg({"ABS_RRATE_%": pd.Series.mean})
# zero_vals.sample(5)

# rel_vals = []
# # Iterate through all rows in plot data frame
# for index, row in rel_plot_df.iterrows():
#     abs_val = row[9] # current recovery rate
#     zero_val = zero_vals.loc[(zero_vals["EXPERIMENT_NAME"] == row[1]) & (zero_vals["TIME"] == row[3]), "ABS_RRATE_%"].values[0] # matching control value 
#     rel_vals.append(round(100 - ((abs_val / zero_val) * 100), 2) * -1 / 2) # append to final list
# plot_df["REL_RRATE_%"] = rel_vals # assign final list to new column in plot data frame
# plot_df = plot_df[plot_df["SAMPLE_NAME"] != "Control"] # filter all control samples (no control-like!)
# plot_df.sample(5)